## 准备数据

In [3]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [4]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [8]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        self.W1 = tf.Variable(tf.random.normal([28*28, 100], stddev=0.01))
        self.b1 = tf.Variable(tf.zeros([100]))
        self.W2 = tf.Variable(tf.random.normal([100, 10], stddev=0.01))
        self.b2 = tf.Variable(tf.zeros([10]))
        ####################
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        x = tf.reshape(x, [-1, 28*28])
        h1 = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)
        logits = tf.matmul(h1, self.W2) + self.b2
        ####################
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [9]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [ ]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 2.2433536 ; accuracy 0.47341666
epoch 1 : loss 2.2428474 ; accuracy 0.47361666
epoch 2 : loss 2.2423375 ; accuracy 0.47373334
epoch 3 : loss 2.2418237 ; accuracy 0.47376665
epoch 4 : loss 2.2413063 ; accuracy 0.47393334
epoch 5 : loss 2.2407846 ; accuracy 0.47408333
epoch 6 : loss 2.240259 ; accuracy 0.4742
epoch 7 : loss 2.2397292 ; accuracy 0.47438332
epoch 8 : loss 2.2391958 ; accuracy 0.4743
epoch 9 : loss 2.2386584 ; accuracy 0.47458333
epoch 10 : loss 2.2381167 ; accuracy 0.47466666
epoch 11 : loss 2.2375708 ; accuracy 0.47495
epoch 12 : loss 2.2370212 ; accuracy 0.47515
epoch 13 : loss 2.2364671 ; accuracy 0.47531667
epoch 14 : loss 2.2359087 ; accuracy 0.4754
epoch 15 : loss 2.2353468 ; accuracy 0.47538334
epoch 16 : loss 2.2347806 ; accuracy 0.47555
epoch 17 : loss 2.23421 ; accuracy 0.47565
epoch 18 : loss 2.233635 ; accuracy 0.47596666
epoch 19 : loss 2.2330558 ; accuracy 0.47603333
epoch 20 : loss 2.2324724 ; accuracy 0.47623333
epoch 21 : loss 2.2318845 ; ac